# Fraud Detection & Transaction Risk Agent

**HackForge FinTech Codeathon Submission**

Financial systems process thousands of transactions, making it difficult to manually identify fraudulent activity. Fraudulent transactions often closely resemble legitimate ones.

**Goal:** Build an AI-powered fraud detection system that analyzes transaction patterns and identifies potentially fraudulent transactions.

## Requirements covered in this notebook
1. Transaction data processing
2. Fraud classification
3. Anomaly detection
4. Fraud probability score
5. Risk-level classification
6. Suspicious transaction identification
7. Explainable prediction

## How to run
Run all cells top to bottom (Runtime → Run all in Colab, or Cell → Run All in Jupyter). Requires: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `joblib`.


## 1. Setup

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib joblib

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Load the Dataset

Upload `transactions_data.csv` to this notebook's file area (10,000 transactions), or in Colab use the file upload cell below.


In [ ]:
# If running in Google Colab, uncomment to upload the file manually:
# from google.colab import files
# uploaded = files.upload()

raw_df = pd.read_csv("transactions_data.csv")
print(f"Loaded {len(raw_df)} transactions")
raw_df.head()


## 3. Requirement 1 — Transaction Data Processing

Clean the data (drop nulls/duplicates) and encode the categorical `merchant_category` column so it can be used by the ML model.


In [ ]:
def process_data(df, label_encoder=None):
    df = df.copy()
    df = df.dropna()
    df = df.drop_duplicates(subset="transaction_id")

    if label_encoder is None:
        label_encoder = LabelEncoder()
        df["merchant_category_encoded"] = label_encoder.fit_transform(df["merchant_category"])
    else:
        df["merchant_category_encoded"] = label_encoder.transform(df["merchant_category"])

    feature_cols = [
        "amount", "hour_of_day", "merchant_category_encoded",
        "location_is_new", "device_is_new", "transactions_last_hour",
    ]
    return df, feature_cols, label_encoder

df, feature_cols, label_encoder = process_data(raw_df)
print(f"Processed {len(df)} transactions. Features used: {feature_cols}")
df.head()


## 4. Requirement 3 — Anomaly Detection

Use an unsupervised `IsolationForest` to flag transactions that are statistically unusual, independent of the labeled fraud data.


In [ ]:
def detect_anomalies(df, feature_cols, iso_forest=None):
    if iso_forest is None:
        iso_forest = IsolationForest(contamination=0.1, random_state=RANDOM_SEED)
        iso_forest.fit(df[feature_cols])
    raw_flags = iso_forest.predict(df[feature_cols])
    df["anomaly_flag"] = (raw_flags == -1).astype(int)
    return df, iso_forest

df, iso_forest = detect_anomalies(df, feature_cols)
print(f"Flagged {df['anomaly_flag'].sum()} statistical anomalies out of {len(df)}")


## 5. Requirement 2 & 4 — Fraud Classification + Probability Score

Fraud is rare (~2.5% of transactions), so a naive model tends to miss real fraud cases even while showing high accuracy. We **oversample the fraud class in the training data** so the model sees enough fraud examples to learn from, and we **lower the decision threshold to 0.3** (instead of the default 0.5) — because in fraud detection, missing real fraud is usually costlier than a false alarm.


In [ ]:
X = df[feature_cols]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# Oversample the minority (fraud) class in the training data only
train_df = X_train.copy()
train_df["is_fraud"] = y_train.values
fraud_rows = train_df[train_df["is_fraud"] == 1]
legit_rows = train_df[train_df["is_fraud"] == 0]
fraud_oversampled = fraud_rows.sample(n=len(legit_rows), replace=True, random_state=RANDOM_SEED)
balanced_train = pd.concat([legit_rows, fraud_oversampled]).sample(frac=1, random_state=RANDOM_SEED)
X_train_bal = balanced_train[feature_cols]
y_train_bal = balanced_train["is_fraud"]

clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED)
clf.fit(X_train_bal, y_train_bal)

y_proba = clf.predict_proba(X_test)[:, 1]
DECISION_THRESHOLD = 0.3
y_pred = (y_proba >= DECISION_THRESHOLD).astype(int)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model test accuracy: {accuracy:.2%}\n")
print("Classification report (decision threshold = 0.3):")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))


### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Fraud"])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig("graphs_confusion_matrix.png", dpi=150)
plt.show()


### ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})", color="#2563eb")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("graphs_roc_curve.png", dpi=150)
plt.show()


### Feature Importance

In [ ]:
importances = clf.feature_importances_
order = np.argsort(importances)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(np.array(feature_cols)[order], importances[order], color="#2563eb")
ax.set_xlabel("Importance")
ax.set_title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.savefig("graphs_feature_importance.png", dpi=150)
plt.show()


## 6. Requirement 5 — Risk-Level Classification

Bucket each transaction's fraud probability into Low / Medium / High risk.


In [ ]:
def assign_risk_level(prob):
    if prob >= 0.7:
        return "High"
    elif prob >= 0.3:
        return "Medium"
    else:
        return "Low"

df["fraud_probability"] = clf.predict_proba(df[feature_cols])[:, 1]
df["risk_level"] = df["fraud_probability"].apply(assign_risk_level)
df["risk_level"].value_counts()


### Risk Level Distribution

In [ ]:
risk_counts = df["risk_level"].value_counts().reindex(["Low", "Medium", "High"])
fig, ax = plt.subplots(figsize=(6, 5))
colors = ["#22c55e", "#f59e0b", "#ef4444"]
ax.bar(risk_counts.index, risk_counts.values, color=colors)
ax.set_ylabel("Number of Transactions")
ax.set_title("Risk Level Distribution (Full Dataset)")
for i, v in enumerate(risk_counts.values):
    ax.text(i, v + max(risk_counts.values) * 0.01, str(v), ha="center")
plt.tight_layout()
plt.savefig("graphs_risk_level_distribution.png", dpi=150)
plt.show()


### Fraud Probability Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.hist(df["fraud_probability"], bins=30, color="#2563eb", edgecolor="white")
ax.set_xlabel("Fraud Probability Score")
ax.set_ylabel("Number of Transactions")
ax.set_title("Distribution of Fraud Probability Scores")
plt.tight_layout()
plt.savefig("graphs_fraud_probability_distribution.png", dpi=150)
plt.show()


## 7. Requirement 7 — Explainable Prediction

Generate a plain-English reason for each transaction's flag, based on which risk factors are present.


In [ ]:
def explain_transaction(row):
    reasons = []
    if row["amount"] > 8000:
        reasons.append("unusually high amount")
    if row["location_is_new"] == 1:
        reasons.append("transaction from a new/unfamiliar location")
    if row["device_is_new"] == 1:
        reasons.append("transaction from a new device")
    if row["hour_of_day"] in [0, 1, 2, 3]:
        reasons.append("occurred at an unusual hour (late night)")
    if row["transactions_last_hour"] > 3:
        reasons.append("multiple transactions in a short time window")
    if row["anomaly_flag"] == 1:
        reasons.append("statistically anomalous pattern detected")
    return "Flagged due to: " + ", ".join(reasons) if reasons else "No strong risk indicators found."

df["explanation"] = df.apply(explain_transaction, axis=1)
df[["transaction_id", "fraud_probability", "risk_level", "explanation"]].head()


## 8. Requirement 6 — Suspicious Transaction Identification

Filter and rank all Medium/High risk transactions — these are the ones that need review.


In [ ]:
suspicious = df[df["risk_level"].isin(["High", "Medium"])].sort_values(
    "fraud_probability", ascending=False
)
print(f"Flagged {len(suspicious)} suspicious transactions out of {len(df)}")
suspicious[["transaction_id", "amount", "fraud_probability", "risk_level", "explanation"]].head(10)


## 9. Save Everything

Save the trained classifier, anomaly detector, and label encoder together into **one model file** (`fraud_detection_model.pkl`), plus the full scored results as a CSV.


In [ ]:
bundle = {
    "classifier": clf,
    "isolation_forest": iso_forest,
    "label_encoder": label_encoder,
    "feature_cols": feature_cols,
}
joblib.dump(bundle, "fraud_detection_model.pkl")
print("Saved fraud_detection_model.pkl")

df.to_csv("fraud_detection_results.csv", index=False)
print("Saved fraud_detection_results.csv")


## 10. Using the Saved Model on New Data (no retraining needed)

```python
bundle = joblib.load("fraud_detection_model.pkl")
clf = bundle["classifier"]
iso_forest = bundle["isolation_forest"]
label_encoder = bundle["label_encoder"]
feature_cols = bundle["feature_cols"]

new_df = pd.read_csv("new_transactions.csv")
new_df, _, _ = process_data(new_df, label_encoder=label_encoder)
new_df, _ = detect_anomalies(new_df, feature_cols, iso_forest=iso_forest)
new_df["fraud_probability"] = clf.predict_proba(new_df[feature_cols])[:, 1]
new_df["risk_level"] = new_df["fraud_probability"].apply(assign_risk_level)
new_df["explanation"] = new_df.apply(explain_transaction, axis=1)
```

## Summary

- **Model:** Random Forest Classifier (200 trees, trained on oversampled data, 0.3 decision threshold) + Isolation Forest for anomaly detection
- **Dataset:** 10,000 transactions
- **Test accuracy:** ~97.5% · **Fraud recall:** ~66% (catches roughly 2 out of every 3 real fraud cases)
- **Output:** `fraud_detection_model.pkl` (single reusable model file) and `fraud_detection_results.csv` (scored transactions)

## Live Demo

A live, interactive demo (`app.py`) is included in this repo — enter a transaction's details and get an instant fraud score, risk level, and explanation. Run it with:
```bash
pip install streamlit
streamlit run app.py
```
Or try the deployed public version (see README for the link).
